<a href="https://colab.research.google.com/github/tracykeeya/nlp_project_group_Sandra_Tracy/blob/main/Copy_of_Group_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets huggingface_hub gradio fastapi uvicorn pypdf sentence-transformers

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")      # read from Colab Secrets, never paste tokens in code
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
print("Logged in as:", HF_USERNAME)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: Tracykee


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

LOCAL_MODEL_DIR = "/content/drive/MyDrive/model_bert"   # TODO: path to your saved model

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOCAL_MODEL_DIR)

clf = pipeline("text-classification", model=model, tokenizer=tokenizer)
print(clf("Terrible experience, I want my money back."))

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_0', 'score': 0.9962285757064819}]


In [ ]:
MODEL_NAME = "model_bert"    # TODO: change to your model
MODEL_ID = f"{HF_USERNAME}/{MODEL_NAME}"

model.push_to_hub(MODEL_ID)
tokenizer.push_to_hub(MODEL_ID)
print("Model page:", f"https://huggingface.co/{MODEL_ID}")
#pushing the model to hugging face

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...aq9uzen/model.safetensors:   3%|2         | 7.97MB /  268MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Model page: https://huggingface.co/Tracykee/model_bert


In [ ]:
#Write a model card
#A model card tells people what your model does and where it fails. Fill in your real numbers.

from huggingface_hub import ModelCard

card_text = f"""---
language: en
pipeline_tag: text-classification
tags: [sentiment-analysis]
---

# {MODEL_NAME}

Sentiment classifier (positive / negative) fine-tuned from `distilbert-base-uncased`.

**Group members:** Gumoshabe Sandra and Keeya Tracy
**Dataset:** stanfordnlp/imdb

## Results on the test set
| Metric | Score |
|---|---|
| Accuracy | 0.9090 |
| F1 | 0.9090 |

## Example predictions
1. "Terrible experience, I want my money back." -> Negative
2. "This is boring" -> Negative
3. "Absolutely wonderful movie with great acting!" -> Positive

## Limitations
Trained on English movie reviews only; struggles with sarcasm and highly complex or mixed language text.
"""

card = ModelCard(card_text)
card.push_to_hub(MODEL_ID)
print("Model card uploaded.")

No files have been modified since last commit. Skipping to prevent empty commit.


Model card uploaded.


In [ ]:
hub_clf = pipeline("text-classification", model=MODEL_ID)
hub_clf("I really enjoyed this class!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'LABEL_1', 'score': 0.997963547706604}]

In [ ]:
 #Build a Gradio demo
#Gradio turns a Python function into a web app. First test it here in Colab.
import gradio as gr

def predict(text):
    if not text.strip():
        return {}
    result = hub_clf(text)[0]
    score = result["score"]
    # Gradio's Label component shows a confidence bar for each class
    other = "NEGATIVE" if result["label"] == "POSITIVE" else "POSITIVE"
    return {result["label"]: score, other: 1 - score}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    description="Type a sentence and see if it is positive or negative.",
    examples=["The food was amazing!", "Worst service ever.", "It was okay, nothing special."],
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a96f4933d501c558a2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%%writefile app.py
import gradio as gr
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id
clf = pipeline("text-classification", model=MODEL_ID)

def predict(text):
    if not text.strip():
        return {}
    r = clf(text)[0]
    other = "NEGATIVE" if r["label"] == "POSITIVE" else "POSITIVE"
    return {r["label"]: r["score"], other: 1 - r["score"]}

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(lines=3, placeholder="Type a sentence..."),
    outputs=gr.Label(num_top_classes=2),
    title="Sentiment Analyser",
    examples=["The food was amazing!", "Worst service ever."],
)
demo.launch()

Overwriting app.py


In [ ]:
%%writefile requirements.txt
transformers
torch

Overwriting requirements.txt


In [ ]:
#from huggingface_hub import HfApi
#api = HfApi()

# Replace the placeholder model id inside app.py with yours
#app_code = open("app.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID)
#open("app.py", "w").write(app_code)

#SPACE_ID = f"{HF_USERNAME}/group1-sentiment-demo"   # TODO: change group number
#api.create_repo(SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
#api.upload_file(path_or_fileobj="app.py", path_in_repo="app.py", repo_id=SPACE_ID, repo_type="space")
#api.upload_file(path_or_fileobj="requirements.txt", path_in_repo="requirements.txt", repo_id=SPACE_ID, repo_type="space")
#print("Your Space (takes 2-5 minutes to build):", f"https://huggingface.co/spaces/{SPACE_ID}")

In [ ]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "YOUR_USERNAME/group1-sentiment-model"   # TODO: your model id

app = FastAPI(title="Sentiment API")

# Load the model ONCE when the app starts, not on every request (that would be very slow)
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))

Overwriting main.py


In [ ]:
# Safely terminate any old server process if running
try:
    server.terminate()
    server.wait()
except NameError:
    pass

# Correctly write main.py with the proper MODEL_ID variable from Colab's state (doubling curly braces to avoid f-string format errors)
main_code = f"""
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import pipeline

MODEL_ID = "{MODEL_ID}"

app = FastAPI(title="Sentiment API")

# Load the model once when the app starts
clf = pipeline("text-classification", model=MODEL_ID)

class TextIn(BaseModel):
    text: str

class PredictionOut(BaseModel):
    label: str
    score: float

@app.get("/health")
def health():
    return {{"status": "ok"}}

@app.post("/predict", response_model=PredictionOut)
def predict(item: TextIn):
    if not item.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty")
    result = clf(item.text)[0]
    return PredictionOut(label=result["label"], score=round(result["score"], 4))
"""

with open("main.py", "w") as f:
    f.write(main_code)

import subprocess, time
# Start uvicorn server in the background
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)   # Give it ample time to download/load the model and spin up
print("API started successfully!")

API started successfully!


In [ ]:
# Put your model id into main.py, then start the API in the background
open("main.py", "w").write(open("main.py").read().replace("YOUR_USERNAME/group1-sentiment-model", MODEL_ID))

import subprocess, time
server = subprocess.Popen(["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"])
time.sleep(25)   # give it time to load the model
print("API started")

API started


In [ ]:
import requests

BASE = "http://127.0.0.1:8000"
print(requests.get(f"{BASE}/health").json())

for sentence in ["I love this app!", "This is so frustrating.", ""]:
    r = requests.post(f"{BASE}/predict", json={"text": sentence})
    print(r.status_code, r.json())

{'status': 'ok'}
200 {'label': 'LABEL_1', 'score': 0.9981}
200 {'label': 'LABEL_0', 'score': 0.9986}
400 {'detail': 'Text cannot be empty'}


In [ ]:
server.terminate()   # stop the API when you are done

**PART B:CHATBOT**


In [ ]:
from transformers import pipeline

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # change if unavailable
llm = pipeline("text-generation", model=LLM_MODEL, torch_dtype="auto", device_map="auto")

def ask_llm(messages, max_tokens=400):
    response = llm(messages, max_new_tokens=max_tokens, do_sample=True, temperature=0.2)
    return response[0]["generated_text"][-1]["content"]

print(ask_llm([
    {"role": "system", "content": "You are a helpful assistant. Answer in one sentence."},
    {"role": "user", "content": "What is natural language processing?"},
]))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Natural Language Processing (NLP) is the branch of artificial intelligence that enables computers to understand, interpret, and generate human language.


In [ ]:
# Compare: your fine-tuned model vs the LLM
#The LLM can do sentiment analysis with no training at all, just a prompt. But is it better, faster, or cheaper?

import time, json
from transformers import pipeline

# Fallback definition if hub_clf is not yet loaded from previous steps
if 'hub_clf' not in globals():
    print("hub_clf was not found. Loading a default fine-tuned sentiment classifier ('distilbert-base-uncased-finetuned-sst-2-english')...")
    hub_clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

test_sentences = [
    "The delivery was quick and the packaging was great.",
    "I waited two hours and nobody helped me.",
    "Not bad, but I expected more for the price.",
    "Absolutely brilliant lecture today!",
    "The app keeps crashing every time I open it.",
]

for s in test_sentences:
    t0 = time.time(); mine = hub_clf(s)[0]["label"]; t_mine = time.time() - t0
    t0 = time.time()
    llm_label = ask_llm([
        {"role": "system", "content": "Classify the sentiment. Reply with ONE word: POSITIVE or NEGATIVE."},
        {"role": "user", "content": s},
    ], max_tokens=5).strip()
    t_llm = time.time() - t0
    print(f"{s[:45]:<47} mine={mine:<9} ({t_mine:.2f}s)  llm={llm_label:<9} ({t_llm:.2f}s)")

The delivery was quick and the packaging was    mine=LABEL_1   (0.01s)  llm=POSITIVE  (0.25s)
I waited two hours and nobody helped me.        mine=LABEL_0   (0.01s)  llm=NEGATIVE  (0.25s)
Not bad, but I expected more for the price.     mine=LABEL_1   (0.01s)  llm=NEGATIVE  (0.24s)
Absolutely brilliant lecture today!             mine=LABEL_1   (0.01s)  llm=POSITIVE  (0.22s)
The app keeps crashing every time I open it.    mine=LABEL_1   (0.01s)  llm=NEGATIVE  (0.26s)


In [ ]:
#Getting structured JSON from an LLM
complaint = "I paid for MTN data bundles yesterday but they never came. This is the third time! Fix it today."

reply = ask_llm([
    {"role": "system", "content": (
        "Extract information from customer complaints. "
        "Reply ONLY with JSON with keys: product, issue, urgency (low, medium or high). No other text."
    )},
    {"role": "user", "content": complaint},
])
print(reply)
data = json.loads(reply.strip().strip("`").removeprefix("json"))
data

{
  "product": "MTN data bundles",
  "issue": "Delivery of purchased products",
  "urgency": "high"
}


{'product': 'MTN data bundles',
 'issue': 'Delivery of purchased products',
 'urgency': 'high'}

**B2 · RAG: answer questions from your PDF**

In [ ]:
from google.colab import files
uploaded = files.upload()
PDF_PATH = list(uploaded.keys())[0]
print("Using:", PDF_PATH)

Saving Keeya Tracy Cover letter SafeBoda.pdf to Keeya Tracy Cover letter SafeBoda.pdf
Using: Keeya Tracy Cover letter SafeBoda.pdf


In [ ]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
print(f"{len(pages)} pages. First 300 characters of page 1:\n")
print(pages[0][1][:300])

2 pages. First 300 characters of page 1:

Keeya
 
Tracy
 
Kampala
 
-
 
Uganda
 
 
keeyatracy@gmail.com
 
+256705724858
 
Portfolio
 
 
 
 
10/01/2026
 
 
The
 
Hiring
 
Manager
 
SafeBoda
 
RE:
 
Application
 
for
 
Web
 
Design
 
and
 
Development
 
Opportunity
 
 
Dear
 
Hiring
 
Manager,
 
I
 
am
 
writing
 
to
 
express
 
my
 
interest


In [ ]:
#Step 1 · Split into chunks (keeping the page number)
def chunk_pages(pages, chunk_size=500, overlap=100):
    chunks = []
    for page_num, text in pages:
        text = " ".join(text.split())          # clean up extra spaces and line breaks
        start = 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if len(piece) > 50:                # skip tiny leftovers
                chunks.append({"page": page_num, "text": piece})
            start += chunk_size - overlap      # overlap so sentences aren't cut in half
    return chunks

chunks = chunk_pages(pages)
print(len(chunks), "chunks")
print(chunks[0])

5 chunks
{'page': 1, 'text': 'Keeya Tracy Kampala - Uganda keeyatracy@gmail.com +256705724858 Portfolio 10/01/2026 The Hiring Manager SafeBoda RE: Application for Web Design and Development Opportunity Dear Hiring Manager, I am writing to express my interest in the opportunity to contribute to the design and development of the proposed SafeBoda e-commerce website. I am a Software Developer with practical experience building web based systems and a strong interest in creating modern, responsive and user friendly digital solut'}


In [ ]:
#Step 2 · Embed the chunks
#sentence-transformers models are trained so that sentences with similar meaning get similar vectors, which makes them better for search than the mean-pooled BERT from the tutorial.
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chunk_vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)
print(chunk_vectors.shape)   # (number of chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(5, 384)


In [ ]:
#Step 3 · Retrieve the most relevant chunks for a question
def retrieve(question, k=3):
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scores = chunk_vectors @ q_vec            # cosine similarity (vectors are normalised)
    top = np.argsort(scores)[::-1][:k]
    return [{**chunks[i], "score": float(scores[i])} for i in top]

question = "What are the initial requirements for the system"   # TODO: a question about YOUR document
for hit in retrieve(question):
    print(f"page {hit['page']}  score={hit['score']:.2f}  {hit['text'][:120]}...")

page 1  score=0.24  nt, categories, product displays, an administrative interface and API integration. Working on this project has strengthe...
page 1  score=0.18  b based systems and a strong interest in creating modern, responsive and user friendly digital solutions. I am currently...
page 1  score=0.09  Keeya Tracy Kampala - Uganda keeyatracy@gmail.com +256705724858 Portfolio 10/01/2026 The Hiring Manager SafeBoda RE: App...


In [ ]:
#Step 4 · Let the LLM answer from those chunks only with guaranteed programmatic page citation injection
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. "
    "If the answer is not in the context, say: I don't know, that is not in the document."
)

def rag_answer(question, k=3):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[Page {h['page']}]: {h['text']}" for h in hits)

    # Get the answer from LLM
    answer = ask_llm([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ])

    # If the LLM doesn't know, return it immediately without adding misleading citations
    if "I don't know" in answer:
        return answer

    # Programmatic Citation Injection:
    # Split answer into lines/sentences and match them back to retrieved source page chunks
    lines = answer.split("\n")
    annotated_lines = []
    for line in lines:
        trimmed = line.strip().lower()
        if not trimmed or len(trimmed) < 10:
            annotated_lines.append(line)
            continue

        # Find which retrieved chunk page best matches this line
        best_page = None
        for h in hits:
            # If there's a significant keyword overlap or direct substring match
            words_in_line = set(w for w in trimmed.split() if len(w) > 3)
            words_in_chunk = set(w for w in h['text'].lower().split() if len(w) > 3)
            if words_in_line and (len(words_in_line & words_in_chunk) / len(words_in_line)) > 0.35:
                best_page = h['page']
                break

        if best_page:
            # Append the page citation directly to the end of the matching bullet point or sentence
            annotated_lines.append(f"{line} (page {best_page})")
        else:
            annotated_lines.append(line)

    return "\n".join(annotated_lines)

print(rag_answer(question))

I don't know, that is not in the document.


In [ ]:
#Test it properly
#Write 5 questions: 3 that your document answers and 2 that it does not answer. A good RAG system says "I don't know" for the last 2 instead of inventing an answer.
test_questions = [
   "what are the main objectives of the project",
   "Which features won`t be part of the first implementation of the project",
   "give examples of existing Ai creative platforms",
    "Who won the 2010 World Cup?",                  # NOT in the document
    "what was the previous song",
]
for q in test_questions:
    print("Q:", q)
    print("A:", rag_answer(q), "\n")

Q: what are the main objectives of the project
A: The main objectives of the project appear to be:

- Developing responsive interfaces (page 1)
- Integrating frontend and backend functionality (page 1)
- Managing data (page 1)
- Creating solutions with users in mind (page 1)
- Building an e-commerce-oriented platform
- Presenting products clearly (page 1)
- Creating intuitive user experiences (page 1)
- Connecting different parts of a web application (page 1) 

Q: Which features won`t be part of the first implementation of the project


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


A: I don't know, that is not in the document. 

Q: give examples of existing Ai creative platforms
A: I don't know, that is not in the document. 

Q: Who won the 2010 World Cup?
A: I don't know, that is not in the document. 

Q: what was the previous song
A: I don't know, that is not in the document. 



In [ ]:
# Turn it into a chatbot
#gr.ChatInterface gives you a chat window. We pass the conversation history to the LLM so it remembers earlier questions (e.g. "What about the second one?")
def chat(message, history):
    hits = retrieve(message)
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in hits)

    messages = [{"role": "system", "content": SYSTEM_PROMPT + " Be polite. Refuse questions unrelated to the document."}]
    messages += history[-6:]                    # last few turns only, to keep it short and cheap
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})
    return ask_llm(messages)

bot = gr.ChatInterface(
    fn=chat,
    title="Question Answer  Assistant",                   # TODO: e.g. "Hostel Rules Assistant"
    examples=["What is this document about?"],
)
bot.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cb7babc43a8a409515.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Run the Gradio Chatbot locally inside Colab
bot.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb7babc43a8a409515.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# Deploy the chatbot to Spaces
#The Space needs your PDF, the app code, and your token stored as a secret (never inside the code).
%%writefile chatbot_app.py
import os
import numpy as np
import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient

PDF_PATH = "document.pdf"
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
SYSTEM_PROMPT = (
    "You answer questions using ONLY the context provided. Cite page numbers like (page 3). "
    "If the answer is not in the context, say: I don't know, that is not in the document. "
    "Be polite. Refuse questions unrelated to the document."
)

llm = InferenceClient(api_key=os.environ["HF_TOKEN"])   # read from the Space secret
embedder = SentenceTransformer("all-MiniLM-L6-v2")

pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(PdfReader(PDF_PATH).pages)]
chunks = []
for page_num, text in pages:
    text = " ".join(text.split())
    for start in range(0, len(text), 400):
        piece = text[start:start + 500]
        if len(piece) > 50:
            chunks.append({"page": page_num, "text": piece})
vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True)

def retrieve(q, k=3):
    scores = vectors @ embedder.encode([q], normalize_embeddings=True)[0]
    return [chunks[i] for i in np.argsort(scores)[::-1][:k]]

def chat(message, history):
    context = "\n\n".join(f"[page {h['page']}] {h['text']}" for h in retrieve(message))
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history[-6:]
    messages.append({"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message}"})
    r = llm.chat.completions.create(model=LLM_MODEL, messages=messages, max_tokens=400, temperature=0.2)
    return r.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages", title="Ask our document").launch()

Writing chatbot_app.py


In [ ]:
%%writefile chatbot_requirements.txt
pypdf
sentence-transformers
huggingface_hub
numpy

Writing chatbot_requirements.txt


In [ ]:
from huggingface_hub import HfApi
api = HfApi()

CHAT_SPACE_ID = f"{HF_USERNAME}/group1-doc-chatbot"   # TODO: change group number

api.create_repo(CHAT_SPACE_ID, repo_type="space", space_sdk="gradio", exist_ok=True)
api.upload_file(path_or_fileobj="chatbot_app.py", path_in_repo="app.py", repo_id=CHAT_SPACE_ID, repo_type="space")
api.upload_file(path_or_fileobj="chatbot_requirements.txt", path_in_repo="requirements.txt", repo_id=CHAT_SPACE_ID, repo_type="space")
api.upload_file(path_or_fileobj=PDF_PATH, path_in_repo="document.pdf", repo_id=CHAT_SPACE_ID, repo_type="space")

# Store the token as a SECRET in the Space (safe: not visible in the code)
api.add_space_secret(CHAT_SPACE_ID, "HF_TOKEN", HF_TOKEN)

print("Your chatbot (takes a few minutes to build):", f"https://huggingface.co/spaces/{CHAT_SPACE_ID}")

HfHubHTTPError: Client error '402 Payment Required' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-6aef49c1-555e2d5022d98ad940bead9d;c2c5ae76-c34c-48a0-bf57-72011ca40aa4)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

Static Spaces are free for everyone, but hosting Gradio and Docker Spaces on free cpu-basic requires a PRO subscription. Subscribe at https://huggingface.co/pro